# 分子動力学計算

In [2]:
from ase.io import read, write

from ase import units
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.md.verlet import VelocityVerlet
from ase.md.npt import NPT
from ase.md import MDLogger

from ase.calculators.lj import LennardJones
from ase.visualize import view
from ase.optimize import BFGS
from ase.phonons import Phonons

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math

<div class="alert-info">

**Goal of this tutorial**  
The goal of this tutoal is to understand the overview of Molecular Dynamics (MD).  
  
The computational target is very simple system, liquid Argon, but the methodology of the time evolution, microscopic analysis, statistical mechanics-based analysis, and computation of transport phoenomena is widely used in cutting-edge MD research.  
  
This Jupyter notebook provides the MD simulation and the post analysis code based on ASE (Atomic Simulation Environment).  
ASE is one of the most famaous and useful library of Molecular Simulation. ASE supports a wide range of MD software (Classical MD: GROMACS, LAMMPS, Amber, ..., DFTMD: VASP, CP2K, Quantum Espresso, ..., and other post analysis code: Pymatgen). Furthermore, cutting-edge computational method like NeuralNetwork Potential has been developed on ASE. Therefore, the analysis code and the know-how of ASE can be used in any molecular simulations.
</div>


## 1. Equation of motion
The most important equation of MD is Equation of motion (Newton's 2nd law) because it connects the movements of a particle with the mechanics: 

$$m\frac{d^2r(t)}{dt^2} = F = - \frac{\partial U(r)}{\partial r},$$

where $m$ is the mass, $r$ is the position, $F$ is the force and $U(r)$ is the potential energy, which depends only on the position of the particle. This equation determine the dynamic evalution of the position if one know the force acting on the particle. However, most cases cannot be solved analytically, so it should be solved numerically. In addition, the accuracy of the potential energy $U(r)$ is also concern. In summary, first of all,  researcher have to choose the suitable methods for the target. 
- Integrator
    - velocity-Verlet
    - Leap-frog
    - (Predictor-corrector method) not major now
- Potential
    - Classical Force field (FF)    / Length: –10 $^3$ nm, Time: –10 $^1$ μs, approximately 100 ns/day
    - Density Functional Theory (DFT) / Length: –10 $^1$ nm
    - Neural Network Potential (NNP) / Length: –10 $^2$ nm

Especially, the choise of potential (energy calculation) is important. Classical force field is fast but low accurate bacause it relys on empirical parameter and function type. On the other hand, DFT has good accurary but the cost is so expensive. To overcome these limitation, many methodology development and application of Neural Network Potential has become active recently. Neural Network Potential (NNP) learn the DFT calculation, allowing it to perform DFT accuracy calculations much faster.


Of course, this ordinary differential equations is second order one, so the following two initial condition is required for the simulation. 
- Initial Condition
    - Position (Configuration)
    - Velocity

The sequence of MD calculations is as follows.  
  
<img src="./images/Procedure.png" width="30%">


## 2. Liquid Argon

In this section, we get the initial structure of molecular dynamics by following procedures:
1. Read the .xyz file as `ase.Atoms` object
   - Keyword: File format, Periodic boundary condition (PBC)
2. Geometry optimization to get stable structure as initial structure of MD

In [ ]:
# System Setup
argon_atoms = read("argon.xyz")   # read the initial configuration from xyz file
# view(argon_atoms)   # visualization
sigma = 3.4 # Angstrom
epsilon = 0.01034 # eV
argon_lj = LennardJones(epsilon=epsilon, sigma=sigma, rc=7.5)
argon_atoms.calc = argon_lj

MD simulations normally use "Periodic Boundary Condition (PBC)". 
- Remove surface effects and treat system as bulk
- Cutoff of Lennard–Jones potential should be less than half length of the lattice length to avoid a particle interacting with its own image

In [9]:
argon_atoms.pbc = True  # set periodic boundary conditions

The geometry optimization is performed. An energy minimization procedure consists of adjusting the coordinates of the atoms that are too close to each other until one of the stopping criteria is reached. ASE implements some optimization algorithm, e.g., BFGS, LBFGS, and FIRE. Here, BFGS is used. 

In [10]:
opt = BFGS(argon_atoms, trajectory="argon_opt.traj")
opt.run(fmax=0.03)  # Criteria is "fmax < 0.03 eV/Angstrom"

      Step     Time          Energy          fmax
BFGS:    0 13:48:05      -10.271093        0.153027
BFGS:    1 13:48:05      -10.284423        0.150786
BFGS:    2 13:48:05      -11.014909        0.082141
BFGS:    3 13:48:05      -11.120179        0.099867
BFGS:    4 13:48:05      -11.142594        0.116740
BFGS:    5 13:48:06      -11.147446        0.119228
BFGS:    6 13:48:06      -11.161460        0.118618
BFGS:    7 13:48:06      -11.190539        0.107716
BFGS:    8 13:48:06      -11.248792        0.085170
BFGS:    9 13:48:06      -11.330670        0.084770
BFGS:   10 13:48:06      -11.417679        0.065407
BFGS:   11 13:48:06      -11.462916        0.078705
BFGS:   12 13:48:06      -11.468804        0.093480
BFGS:   13 13:48:06      -11.471307        0.094723
BFGS:   14 13:48:06      -11.479790        0.091651
BFGS:   15 13:48:06      -11.494462        0.082687
BFGS:   16 13:48:06      -11.530018        0.060521
BFGS:   17 13:48:06      -11.588552        0.036922
BFGS:   18 13:

True

In [11]:
# visualize the optimization
opttrj = read("argon_opt.traj", index=":")
view(opttrj)

<Popen: returncode: None args: ['/Users/srak/miniconda3/envs/md/bin/python',...>

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/srak/miniconda3/envs/md/lib/python3.12/site-packages/ase/gui/pipe.py", line 32, in <module>
    main()
  File "/Users/srak/miniconda3/envs/md/lib/python3.12/site-packages/ase/gui/pipe.py", line 28, in main
    plt.show()
  File "/Users/srak/miniconda3/envs/md/lib/python3.12/site-packages/matplotlib/pyplot.py", line 612, in show
    return _get_backend_mod().show(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/srak/miniconda3/envs/md/lib/python3.12/site-packages/matplotlib_inline/backend_inline.py", line 90, in show
    display(
  File "/Users/srak/miniconda3/envs/md/lib/python3.12/site-packages/IPython/core/display_functions.py", line 265, in display
    print(*objs)
ValueError: I/O operation on closed file.


<div class="alert alert-danger">

***Caution!!!!!!!***  

WSL環境だと、最新版version 3.23.0のASEでvisualize バグがある  
解決策は下記URL参照  
https://gitlab.com/ase/ase/-/issues/1511#:~:text=def%20set_windowtype(win,type%27%2C%20wmtype)%0A%20%20%20%20pass
</div>

## 3. _NVE_ integrator (velocity-Verlet)

The time integrator called "**velocity-Verlet**" method is implemented on ASE. 

$$
v\left( t+\frac{\Delta t}{2} \right) \gets v(t) + \frac{F(t)}{m}\frac{\Delta t}{2} + \mathcal{O}((\Delta t)^3), 
$$

$$
r\left(t+\Delta t \right) \gets r(t) + v\left(t + \frac{\Delta t}{2} \right) \Delta t + \mathcal{O}((\Delta t)^3)
$$

$$
v\left( t+\Delta t \right) \gets v\left( t+\frac{\Delta t}{2} \right) + \frac{F(t+\Delta t)}{m} \frac{\Delta t}{2} + \mathcal{O}((\Delta t)^3)
$$

----
To solve the equation of motion, initial velocities are also required.  
This tutorial of liquid Argon, the temperature would like to be set as 94.4 K. For efficient equilibration, the initial velocity should be determined by the target temperature because temperature is related to velocity given by

$$
\left\langle \sum_{i}^N \frac{1}{2}m_i v_i^2 \right\rangle = \frac{3}{2} N k_{\rm{B}}T .
$$

In this tutorial, the initial velocity is determined by Maxwell-Boltzmann distribution as

$$
f(v_{x}) = \sqrt{\frac{m}{2\pi k_{\rm{B}}T} \exp \left( \frac{1}{2}m v_x^2 \right)}
$$

In [12]:
MaxwellBoltzmannDistribution(argon_atoms, temperature_K= 94.4, force_temp=True, rng=np.random.seed(123))
print("Temperature = ", argon_atoms.get_temperature())
print("Total momentum = ", np.sum(argon_atoms.get_momenta(),axis=0))

Temperature =  94.4
Total momentum =  [ 0.46753709 -2.51519731 -3.55659957]


<div class="alert alert-danger">

***Caution!!!!!!!***  
This initial velocity leads to undesired artifact on the MD simulation because total momentum is not zero.  
If total momentum is not zero, the center of mass of system (total system) drifts.  
To avoid this, `Stationary` should be used and confirm that the total momentum is zero.
</div>

In [13]:
Stationary(argon_atoms)
print("Total momentum = ", np.sum(argon_atoms.get_momenta(),axis=0))
print("Temperature = ", argon_atoms.get_temperature())

Total momentum =  [-4.27435864e-15  1.33226763e-15  2.55351296e-15]
Temperature =  94.40000000000003


### *NVE* ensemble (Microcanonical ensemble)
***NVE*** means constant  <u>***N***</u>umber of particles, <u>***V***</u>olume, and <u>***E***</u>energy.  
From the next cell, set the conditions for numerical integration and execute it.  
Important parameter is timestep $\Delta t$.  
  
If $\Delta t$ is too large, then the energy conservation become sick and miss important events.  
If $\Delta t$ is too small, obtaining sufficient statistical average takes very long computational time.

  
Tips: Vibration of Light atom H is most fastest dynamics $\to$ There are some methods to constrain the H bond. (SHAKE, LINKS)
  
  
This calculation (Liquid Argon) uses 5 fs as timestep and this tutorial confirm that it is enough small later.

To perform velocity-Verlet (NVE ensemble), ASE gives `ase.md.verlet.VelocityVerlet`.

In [15]:
integrator = VelocityVerlet(argon_atoms,   # Atoms object
                            5 * units.fs,  # Timestep
                            trajectory="md_nve.traj", # Name of the trajectory file
                            loginterval=10)  # Output interval of trajectoryfile
nsteps = 500

# Setting of logfile
integrator.attach(MDLogger(integrator, argon_atoms, logfile="md_nve.log", header=True, stress=False,
                mode="w"), interval=10)
def progress():
    import datetime
    global initial_timestamp
    global initial_step
    try: 
        time_consumed = datetime.datetime.now() - initial_timestamp
        # 1stepあたりにかかった実時間 realtime_per_step
        realtime_per_step = time_consumed.total_seconds() / (integrator.get_number_of_steps() - initial_step)
        # 残り時間
        remaining_time = (nsteps - integrator.get_number_of_steps()) * realtime_per_step
        print(f"Step: {integrator.get_number_of_steps()} / {nsteps}, T: {argon_atoms.get_temperature():.2f} K, Remaining time: {remaining_time:.2f} s")
    except:
        initial_timestamp = datetime.datetime.now()
        initial_step = integrator.get_number_of_steps()
        print(f"Step: {integrator.get_number_of_steps()} / {nsteps}, T: {argon_atoms.get_temperature():.2f} K")

integrator.attach(progress, interval=100)

In [16]:
# Execute MD
try:
    del globals()["initial_timestamp"]
    del globals()["initial_step"]
except:
    pass
integrator.run(nsteps)

Step: 0 / 500, T: 94.40 K
Step: 100 / 500, T: 59.73 K, Remaining time: 2.21 s
Step: 200 / 500, T: 59.15 K, Remaining time: 1.65 s
Step: 300 / 500, T: 60.55 K, Remaining time: 1.10 s
Step: 400 / 500, T: 64.61 K, Remaining time: 0.57 s
Step: 500 / 500, T: 62.49 K, Remaining time: 0.00 s


True

In [ ]:
mdtraj = read("md_nve.traj", index=":")
view(mdtraj)

In [ ]:
df = pd.read_csv("md_nve.log", sep=r'\s+')
df

In [ ]:
df.describe()

In [ ]:
df.plot(subplots=True, layout=(3,2), x="Time[ps]",figsize=(10,6))

<div class="alert alert-success">
Q1. Check the behavior as the timestep is changed
</div>

The temperature is not controlled at the target temperature = 94.4 K.  
Therefore, temperature control method, i.e., NVT ensemble simulation is necessary.  

## 4. _NVT_ integrator (velocity-Verlet + Nose-Hoover thermostat)

There are many ways to control the system temperature.  
- Berendsen thermostat
- Nose-Hoover thermostat
- Langevin thermostat
- ....  

One of the most commonly used algorithm is **Nose-Hoover thermostat**. Advantage of Nose-Hoover thermostat is that system follows canonical ensemble (*NVT* ensemble). ***NVT*** means constant  <u>***N***</u>umber of particles, <u>***V***</u>olume, and <u>***T***</u>empearture.  
  
Under the canonical (*NVT*) ensemble, the distribution function $f(\boldsymbol{q}, \boldsymbol{p})$ must follow as below relationship:  

$$
f(\boldsymbol{q}, \boldsymbol{p}) = \frac{1}{N!h^{3N}} \frac{\exp(-\beta \mathcal{H}(\boldsymbol{q}, \boldsymbol{p}))}{Z},  \tag{5.1}  
$$


$$
\mathrm{s.t.}\;  Z = \frac{1}{N!h^{3N}}\int \rm{d}{\boldsymbol{q}}\int \rm{d}{\boldsymbol{p}} \exp \left[ -\beta \mathcal{H} \right], 
$$

$$
\beta = \frac{1}{k_{\rm{B}}T},
$$

where $\boldsymbol{q}$, $\boldsymbol{p}$, $N$, $h$, $k_{\rm{B}}$, and $T$ are position, momentum, number of particles, Planck constant, Boltzmann constant, and temperature, respectively. $Z$ is partition function and $\beta$ is inverse temperature. Under the canonical ensemble, one physical property $A(\boldsymbol{q}, \boldsymbol{p})$ is obtained as follows:  

$$
\langle A \rangle _{NVT} = \int \rm{d}{\boldsymbol{q}}\int \rm{d}{\boldsymbol{p}} \; \it{A}(\boldsymbol{q}, \boldsymbol{p})f(\boldsymbol{q}, \boldsymbol{p}).   \tag{5.2}
$$

By using Nose-Hoover thermostat, system follows the Eq.(5.1). Thus, if the simulation time is enough long, the time averaged $ A(\boldsymbol{q}, \boldsymbol{p}) $ is equal to Eq.(5.2):  

$$
\langle A \rangle _{NVT} = \langle A \rangle _{\rm{time, Nose-Hoorver}} = \frac{\sum_{k}^M A(t_{k})}{M}
$$ 

The equation of motion with Nose-Hoover thermostat is given by
$$\dot{q_i} = \frac{p_i(t)}{m_i} , $$

$$\dot{p_i} = F_i(t) - \zeta p_i   ,$$

$$\dot{\zeta} = \frac{1}{\tau_{T}^2} \left[ \frac{\mathcal{T}(t)}{T_0s} - 1 \right] , $$

$$\mathrm{s.t.} \; \mathcal{T}(t) = \frac{1}{3 Nk_{\rm{B}}} \sum_{i}^N m_iv_{i}^2  ,$$

where $\mathcal{T}(t)$ is called instantaneous temperature and $\tau_{T}$ is the time constant of Nose-Hoover thermostat.  
`Notice`: Instantaneous temperature has no physical meaning. Only the ensemble averaged of $\mathcal{T}(t)$, i.e., $\langle \mathcal{T} \rangle = T $ (Temperature) has physical meaning.  
Users should decide proper $\tau_{T}$. According to the [ASE documents](https://wiki.fysik.dtu.dk/ase/ase/md.html), the good choice of $\tau_{T}$ (parameter name: ttime) is 25 fs. 

In [ ]:
# argon_atoms = mdtraj[-1].copy()
# argon_atoms.calc = argon_lj

integrator = NPT(argon_atoms, 5 * units.fs, temperature_K = 94.4, ttime = 25 * units.fs, pfactor = None, trajectory="md_nvt.traj", loginterval=10) # NH
integrator.attach(MDLogger(integrator, argon_atoms, logfile="md_nvt.log", header=True, stress=False,
                mode="w"), interval=10)
integrator.run(50000)

In [ ]:
mdtraj = read("md_nvt.traj", index=":")
view(mdtraj)

In [ ]:
df = pd.read_csv("md_nvt.log", sep=r'\s+')
df

In [ ]:
df.describe()

In [ ]:
df.plot(subplots=True, layout=(3,2), x="Time[ps]",figsize=(10,6))

<div class="alert alert-success">
Q. Calculate the temperature from 50 ps.  
</div>